# 🎙️ CloneCinema Voice Studio — Multi-Character Kokoro TTS en Google Colab
### Voces Ultra-Realistas en Español para Narración y Diálogos entre Personajes
---
**Roles y Personajes soportados:**
- 🎬 **Narrador:** Tono cinematográfico, documental, neutro.
- 🥔 **Adulto Masculino (ej. Papá Patata):** Voz madura, cálida, protectora.
- 🧅 **Adulta Femenina (ej. Mamá Cebolla):** Expresiva, dulce, maternal, emotiva.
- 🍟 **Niño (ej. Patatín Jr.):** Voz infantil, aguda, juguetona.
- 🍓 **Niña / Adolescente (ej. Doña Fresa):** Tono brillante, alegre y dinámico.

**Requisitos en Colab:**
- **Entorno:** CPU estándar o GPU T4 (en CPU genera en ~0.4s, en GPU en ~0.1s).
- Pulsa **Entorno de ejecución -> Ejecutar todas** y se enlazará automáticamente a CloneCinema.

In [ ]:
# ⚡ [1/3] Instalar dependencias esenciales
import os, sys, subprocess

print('⚡ [1/3] Instalando Kokoro TTS, FastAPI y Cloudflared...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kokoro', 'soundfile', 'fastapi', 'uvicorn', '-q'], check=True)

if not os.path.exists('cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', 'cloudflared'], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

print('✅ Dependencias listas.')

In [ ]:
# 🧠 [2/3] Inicializar motor Kokoro TTS y mapeo de voces multi-personaje
from kokoro import KPipeline
import soundfile as sf
import numpy as np
import io, base64

print('🧠 [2/3] Cargando modelos de voz...')
try:
    pipeline = KPipeline(lang_code='e') # 'e' incluye soporte español y multilingual
except Exception as e:
    print(f'Fallback pipeline init: {e}')
    pipeline = KPipeline(lang_code='a')

VOICE_MAP = {
    'narrator': 'em_santa',
    'adult_male': 'em_alex',
    'adult_female': 'ef_dora',
    'child_boy': 'em_george',
    'child_girl': 'ef_bella'
}

def synthesize_audio(text: str, role: str = 'narrator', speed: float = 1.0):
    voice_key = VOICE_MAP.get(role, 'em_alex')
    generator = pipeline(text, voice=voice_key, speed=speed, split_pattern=r'\n+')
    all_audio = []
    for _, _, audio in generator:
        all_audio.append(audio)
    
    if not all_audio:
        raise ValueError('No se generó audio.')
    
    combined = np.concatenate(all_audio)
    out_buf = io.BytesIO()
    sf.write(out_buf, combined, 24000, format='WAV')
    return base64.b64encode(out_buf.getvalue()).decode('utf-8')

print('✅ Motor de voz y mapeo de personajes configurados.')

In [ ]:
# 🌐 [3/3] Levantar API FastAPI, abrir túnel Cloudflare y auto-registrar en CloneCinema
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn, threading, subprocess, time, re, urllib.request, json

app = FastAPI(title='CloneCinema Voice Studio')

class VoiceRequest(BaseModel):
    text: str
    role: str = 'narrator'
    character_name: str = None
    gender: str = None
    speed: float = 1.0

@app.get('/health')
def health():
    return {'status': 'ok', 'service': 'CloneCinema Voice Studio', 'roles': list(VOICE_MAP.keys())}

@app.post('/synthesize')
def synthesize(req: VoiceRequest):
    try:
        role = req.role
        if not role and req.gender:
            role = 'adult_female' if req.gender == 'female' else 'adult_male'
        b64 = synthesize_audio(req.text, role=role, speed=req.speed)
        return {'success': True, 'audio_base64': b64, 'mime_type': 'audio/wav', 'sample_rate': 24000}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

tunnel_log = 'voice_tunnel.log'
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=open(tunnel_log, 'w'),
    stderr=subprocess.STDOUT
)

print('⏳ Abriendo túnel Cloudflare...')
tunnel_url = None
for _ in range(25):
    time.sleep(2)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print('\n' + '='*70)
    print('🎉 ¡CLONECINEMA VOICE STUDIO EN VIVO!')
    print(f'👉 TÚNEL: {tunnel_url}')
    print('='*70 + '\n')
    
    print('📡 Conectando con CloneCinema Gateway...')
    try:
        reg_payload = json.dumps({'tunnelUrl': tunnel_url}).encode('utf-8')
        req = urllib.request.Request(
            'https://node-clonecinema.vercel.app/api/voice/register',
            data=reg_payload,
            headers={'Content-Type': 'application/json'},
            method='POST'
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            res_data = json.loads(resp.read().decode('utf-8'))
            if res_data.get('success'):
                print('✅ ¡ENLAZADO CON ÉXITO! CloneCinema generará voces de personajes por Colab.')
    except Exception as e:
        print(f'⚠️ Registro automático aviso: {e}')
    
    try:
        hb = 0
        while True:
            time.sleep(30)
            hb += 1
            if hb % 5 == 0:
                try:
                    urllib.request.urlopen(
                        urllib.request.Request('https://node-clonecinema.vercel.app/api/voice/register', data=reg_payload, headers={'Content-Type': 'application/json'}, method='POST'),
                        timeout=5
                    )
                except:
                    pass
    except KeyboardInterrupt:
        print('🛑 Servidor detenido.')
else:
    print('⚠️ No se capturó la URL del túnel. Revisa voice_tunnel.log')
